# CIFAR-10 — Comparaison des optimiseurs
**EFREI Paris — Optimisation Convexe**

Comparaison de 5 optimiseurs : SGD, SGD+Momentum, AdaGrad, RMSProp, Adam

## 1. Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

PyTorch: 2.11.0
Device : cpu


## 2. Configuration

In [3]:
SEED       = 42
NUM_EPOCHS = 10
BATCH_SIZE = 128

torch.manual_seed(SEED)
np.random.seed(SEED)

os.makedirs("results", exist_ok=True)

# (name, optimizer class, kwargs)
OPTIMIZERS = [
    ("SGD",          optim.SGD,     {"lr": 0.01}),
    ("SGD+Momentum", optim.SGD,     {"lr": 0.01, "momentum": 0.9}),
    ("AdaGrad",      optim.Adagrad, {"lr": 0.01}),
    ("RMSProp",      optim.RMSprop, {"lr": 0.001, "alpha": 0.9}),
    ("Adam",         optim.Adam,    {"lr": 0.001, "betas": (0.9, 0.999)}),
]

COLORS = {
    "SGD": "#e41a1c",
    "SGD+Momentum": "#ff7f00",
    "AdaGrad": "#4daf4a",
    "RMSProp": "#377eb8",
    "Adam": "#984ea3",
}

print("Config OK")

Config OK


## 3. Données CIFAR-10

In [4]:
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train : {len(trainset):,} images")
print(f"Test  : {len(testset):,} images")

/Users/thopham/Documents/EFREI2025/ConvexOpt/Convex-Optimisation/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train : 50,000 images
Test  : 10,000 images


## 4. Architecture CNN

### Simple image classification

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # bloc 1 — 32x32
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 16x16
            # bloc 2 — 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 8x8
            # bloc 3 — 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 4x4
            # classifieur
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


# vérification rapide
dummy = SimpleCNN().to(DEVICE)
out   = dummy(torch.randn(4, 3, 32, 32).to(DEVICE))
n_params = sum(p.numel() for p in dummy.parameters())
print(f"Output shape : {out.shape}  (attendu [4, 10])")
print(f"Paramètres   : {n_params:,}")
del dummy

Output shape : torch.Size([4, 10])  (attendu [4, 10])
Paramètres   : 667,178


## 5. Entraînement des 5 optimiseurs

Boucle d'entraînement inline, pas de fonctions intermédiaires.

In [ ]:
all_results = {}
criterion   = nn.CrossEntropyLoss()
epochs_list = list(range(1, NUM_EPOCHS + 1))

for opt_name, opt_cls, opt_kwargs in OPTIMIZERS:
    print(f"\n{'='*50}")
    print(f"Optimiseur : {opt_name}  —  {opt_kwargs}")
    print(f"{'='*50}")

    torch.manual_seed(SEED)
    model     = SimpleCNN().to(DEVICE)
    optimizer = opt_cls(model.parameters(), **opt_kwargs)

    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []
    grad_norms = []

    for epoch in epochs_list:

        # ── Entraînement ──────────────────────────────────
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        batch_grad_norms = []

        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()

            # norme du gradient
            g = sum(p.grad.norm().item() ** 2
                    for p in model.parameters() if p.grad is not None) ** 0.5
            batch_grad_norms.append(g)

            optimizer.step()

            run_loss += loss.item() * images.size(0)
            correct  += (outputs.argmax(1) == labels).sum().item()
            total    += images.size(0)

        train_losses.append(run_loss / total)
        train_accs.append(100.0 * correct / total)
        grad_norms.append(float(np.mean(batch_grad_norms)))

        # ── Évaluation test ───────────────────────────────
        model.eval()
        run_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                run_loss += criterion(outputs, labels).item() * images.size(0)
                correct  += (outputs.argmax(1) == labels).sum().item()
                total    += images.size(0)

        test_losses.append(run_loss / total)
        test_accs.append(100.0 * correct / total)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>2}/{NUM_EPOCHS} "
                  f"| train {train_accs[-1]:.1f}% "
                  f"| test  {test_accs[-1]:.1f}%")

    all_results[opt_name] = {
        "train_loss": train_losses,
        "train_acc":  train_accs,
        "test_loss":  test_losses,
        "test_acc":   test_accs,
        "grad_norm":  grad_norms,
    }
    print(f"  → Meilleure précision test : {max(test_accs):.2f}%")

print("\nToutes les expériences terminées.")


Optimiseur : SGD  —  {'lr': 0.01}
  Epoch  1/10 | train 39.5% | test  49.3%
  Epoch  5/10 | train 66.6% | test  68.0%


## 6. Courbes de convergence

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Comparaison des optimiseurs — CIFAR-10", fontsize=14, fontweight="bold")

metrics = [
    ("train_loss", "Perte (entraînement)", "Loss"),
    ("test_loss",  "Perte (test)",         "Loss"),
    ("train_acc",  "Précision (entraînement)", "Accuracy (%)"),
    ("test_acc",   "Précision (test)",          "Accuracy (%)"),
]

for ax, (metric, title, ylabel) in zip(axes.flat, metrics):
    for name, res in all_results.items():
        ax.plot(epochs_list, res[metric], label=name, color=COLORS[name], linewidth=2)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : results/learning_curves.png")

## 7. Norme du gradient

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for name, res in all_results.items():
    ax.plot(epochs_list, res["grad_norm"], label=name, color=COLORS[name], linewidth=2)

ax.set_title("Évolution de la norme du gradient", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("||∇θ L||")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/grad_norms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : results/grad_norms.png")

## 8. Écart de généralisation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for name, res in all_results.items():
    gap = [tr - te for tr, te in zip(res["train_acc"], res["test_acc"])]
    ax.plot(epochs_list, gap, label=name, color=COLORS[name], linewidth=2)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Écart de généralisation (train acc − test acc)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Gap (%)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/generalization_gap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : results/generalization_gap.png")

## 9. Tableau récapitulatif

In [ ]:
rows = []
for name, res in all_results.items():
    gap  = res["train_acc"][-1] - res["test_acc"][-1]
    ep80 = next((i + 1 for i, a in enumerate(res["test_acc"]) if a >= 80), None)
    rows.append({
        "Optimiseur":          name,
        "Train acc (%)": f"{res['train_acc'][-1]:.2f}",
        "Test acc (%)":  f"{max(res['test_acc']):.2f}",
        "Gap génér. (%)": f"{gap:.2f}",
        "Epochs → 80%":  ep80 if ep80 else f">{NUM_EPOCHS}",
    })

df = pd.DataFrame(rows).set_index("Optimiseur")
display(df)

df.to_csv("results/summary_table.csv")
print("Sauvegardé : results/summary_table.csv")